In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from sklearn.preprocessing import MinMaxScaler
print("Packahges imported")

Packahges imported


In [3]:
# 1. Download Baseline Data via yfinance
print("Downloading Nifty 50 data...")
nifty_df = yf.download("^NSEI", start="2018-01-01", progress=False)

# Standardize multi-index columns if they exist
if isinstance(nifty_df.columns, pd.MultiIndex):
    nifty_df.columns = nifty_df.columns.get_level_values(0)

# 2. Feature Engineering
def calculate_technical_indicators(data):
    df = data.copy()
    
    # Target is predicting the daily return, not the raw price
    df["Return"] = df["Close"].pct_change()
    
    # Moving Averages
    df["SMA_10"] = df["Close"].rolling(window=10).mean()
    df["SMA_50"] = df["Close"].rolling(window=50).mean()
    
    # RSI (14 period)
    delta = df["Close"].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / (loss + 1e-9)
    df["RSI"] = 100 - (100 / (1 + rs))
    
    # MACD
    ema_12 = df["Close"].ewm(span=12, adjust=False).mean()
    ema_26 = df["Close"].ewm(span=26, adjust=False).mean()
    df["MACD"] = ema_12 - ema_26
    
    # What the model needs to guess: Tomorrow's return
    df["Target_Return"] = df["Return"].shift(-1)
    
    return df.dropna()

nifty_data = calculate_technical_indicators(nifty_df)

# 3. Data Splitting (Chronological, no shuffling)
split_date = "2024-01-01"
features = ["Return", "SMA_10", "SMA_50", "RSI", "MACD", "Volume"]

train_mask = nifty_data.index < split_date
test_mask = nifty_data.index >= split_date

X_train = nifty_data.loc[train_mask, features]
y_train = nifty_data.loc[train_mask, "Target_Return"]
X_test = nifty_data.loc[test_mask, features]
y_test = nifty_data.loc[test_mask, "Target_Return"]
test_df = nifty_data.loc[test_mask]

# Scale features strictly on training data
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Train Model A (Top-Down Baseline)
print("Training Baseline Model...")
model_A = xgb.XGBRegressor(n_estimators=100, max_depth=4, random_state=42)
model_A.fit(X_train_scaled, y_train)

# 5. Predict and Evaluate
pred_returns_A = model_A.predict(X_test_scaled)

# Convert predicted returns back into actual index price points
actual_close_t = test_df["Close"].values
actual_close_next = test_df["Close"].shift(-1).dropna().values

pred_price_A = (actual_close_t[:-1]) * (1 + pred_returns_A[:-1])

rmse_A = np.sqrt(mean_squared_error(actual_close_next, pred_price_A))
mape_A = mean_absolute_percentage_error(actual_close_next, pred_price_A) * 100

print("\n--- MODEL A (Top-Down Baseline) RESULTS ---")
print(f"RMSE (Point Error): {rmse_A:.2f}")
print(f"MAPE (Percentage Error): {mape_A:.2f}%")
results_df = pd.DataFrame({"Actual": actual_close_next, "Pred_TopDown": pred_price_A}, index=test_df.index[:-1])
results_df.to_csv("top_down_results.csv")
print("Saved Model A predictions to 'top_down_results.csv'")

Training Baseline Model...

--- MODEL A (Top-Down Baseline) RESULTS ---
RMSE (Point Error): 228.20
MAPE (Percentage Error): 0.71%
Saved Model A predictions to 'top_down_results.csv'


/Users/aryanjain/Desktop/Subject/AIML/research/venv/lib/python3.13/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/Users/aryanjain/Desktop/Subject/AIML/research/venv/lib/python3.13/site-packages/yfinance/utils.py:514: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
/Users/aryanjain/Desktop/Subject/AIML/research/venv/lib/python3.13/site-packages/yfinance/utils.py:518: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for m

model training

In [4]:
import os
import joblib

os.makedirs("artifacts", exist_ok=True)

baseline_artifact = {
    "model_name": "Top-Down (Baseline)",
    "ticker": "^NSEI",
    "model": model_A,
    "scaler": scaler,
    "features": features,
}

joblib.dump(baseline_artifact, "artifacts/baseline_model.joblib")
print("Saved Top-Down Baseline model & scaler to artifacts/baseline_model.joblib")

Saved Top-Down Baseline model & scaler to artifacts/baseline_model.joblib
